If sex has been activated in the command line argument, we use SLiM's initializeSex function to allow sexual reproduction.

In [ ]:
if (sex) {
    initializeSex();
}

If ben has been activated in the command line, we initialize both deleterious and beneficial mutations in mitochondrial and nuclear genomes. The deleterious nuclear mutations (m1 and m2) have a dominance coefficient of 0.5 and selection coefficients are sampled from a gamma distribution with a mean of -0.01 and a beta parameter of 0.2. The beneficial nuclear mutations (m3 and m4) have a dominance coefficient of 0.5 and selection coefficients are sampled from a gamma distribution with a mean of 0.001 and a beta parameter of 1. The deleterious mitochondrial mutations (m5 and m6) have a dominance coefficient of 1.0 and selection coefficients are sampled from a gamma distribution with a mean of -0.05 and a beta parameter of 0.2. The beneficial mitochondrial mutations (m7 and m8) have a dominance coefficient of 1.0 and selection coefficients are sampled from a gamma distribution with a mean of 0.001 and a beta parameter of 1.

In [ ]:
if (ben) {
    // Nuclear mutations
    initializeMutationType("m1", 0.5, "g", -0.01, 0.2); // Nuclear negative A
    initializeMutationType("m2", 0.5, "g", -0.01, 0.2); // Nuclear negative B
    initializeMutationType("m3", 0.5, "g", 0.001, 1); // Nuclear positive A
    initializeMutationType("m4", 0.5, "g", 0.001, 1); // Nuclear positive B
    
    // Mitochondrial mutations
    initializeMutationType("m5", 1.0, "g", -0.05, 0.2); // Mitochondrial negative A
    initializeMutationType("m6", 1.0, "g", -0.05, 0.2); // Mitochondrial negative B
    initializeMutationType("m7", 1.0, "g", 0.001, 1); // Mitochondrial positive A
    initializeMutationType("m8", 1.0, "g", 0.001, 1); // Mitochondrial positive B
}

If ben has not been activated in the command line, nuclear and mitochondrial mutation types are initialized the same way, but all are deleterious.

In [ ]:
else {
    initializeMutationType("m1", 0.5, "g", -0.01, 0.2); // Nuclear negative A
    initializeMutationType("m2", 0.5, "g", -0.01, 0.2); // Nuclear negative B
    initializeMutationType("m3", 0.5, "g", -0.01, 0.2); // Nuclear negative A
    initializeMutationType("m4", 0.5, "g", -0.01, 0.2); // Nuclear negative B
    
    // Mitochondrial mutations
    initializeMutationType("m5", 1.0, "g", -0.05, 0.2); // Mitochondrial negative A
    initializeMutationType("m6", 1.0, "g", -0.05, 0.2); // Mitochondrial negative B
    initializeMutationType("m7", 1.0, "g", -0.05, 0.2); // Mitochondrial negative A
    initializeMutationType("m8", 1.0, "g", -0.05, 0.2); // Mitochondrial negative B
}

We set convertToSubstitution for each mutation type to false to ensure that even after a mutation becomes fixed in the population, it still contributes to overall fitness. This is necessary to account for epistatic interactions.

In [ ]:
// Set convertToSubstitution to F for all mutation types - allows epistasis
m1.convertToSubstitution = F;
m2.convertToSubstitution = F;
m3.convertToSubstitution = F;
m4.convertToSubstitution = F;
m5.convertToSubstitution = F;
m6.convertToSubstitution = F;
m7.convertToSubstitution = F;
m8.convertToSubstitution = F;

We initialize genomic element types g1 and g2 for the nuclear and mitochondrial genomes respectively. In the nuclear genome, m1, m2, m3, and m4 occur at rates of 0.49, 0.49, 0.01, and 0.01 respectively. If ben is activated, m3 and m4 will consist of rare beneficial mutations. Otherwise, all mutations will be deleterious. The same principle applies for the mitochondrial genomic element and its mutations.

In [ ]:
// Genomic element types
initializeGenomicElementType("g1", c(m1, m2, m3, m4), c(0.49, 0.49, 0.01, 0.01)); // Nuclear genome
initializeGenomicElementType("g2", c(m5, m6, m7, m8), c(0.49, 0.49, 0.01, 0.01)); // Mitochondrial genome

We initialize the nuclear chromosome with a length of 1,000,000 and type autosomal. We initialized a mutation rate of 1e-7 with a recombination rate depending on if sex has been activated or not. If sex has been activated, we initialize a recombination rate of 1e-8. Otherwise, the recombination rate is initialized to 0.

In [ ]:
// Nuclear chromosome
initializeChromosome(1, 1e6, type="A", symbol="N"); // autosomal
initializeGenomicElement(g1, 0, 999999);
initializeMutationRate(1e-7);

if (sex) {
    initializeRecombinationRate(1e-8); // recombination rate of 1e-8 per base pair per generation for sexual reproduction
}
else {
    initializeRecombinationRate(0);
}

We initialize the mitochondrial chromsome with a length of 10,000 and type haploid female inheritance. We initialize it with a recombination rate of 0 and a mutation rate of 1e-6, which is 10 times higher than the nuclear mutation rate.

In [ ]:
// Mitochondrial chromosome
initializeChromosome(2, 1e4, type="HF", symbol="M"); // haploid female inheritance
initializeRecombinationRate(0);
initializeGenomicElement(g2, 0, 9999);
initializeMutationRate(1e-6);

We create the population of 250 individuals. If sex has not been activated, we set the cloning rate to 1.0. Otherwise, this cloning rate is kept at the default value of 0.

In [ ]:
1 early() {

	sim.addSubpop("p1", 250); // population of 250 individuals

	if (!sex) {
		p1.setCloningRate(1.0); // cloning rate of 1.0 for asexual reproduction
	}
}

At the end of every generation, we loop through each individual in the population to calculate fitness. We begin by extracting the unique mutations in the nuclear and mitochondrial genomes for each individual.

In [ ]:
1: late() {

	inds = sim.subpopulations.individuals;
	popFitt = c();

	// loop through each individual in the population to calculate fitness
	for (ind in inds) {
		globalFitness = 0;
		unique = ind.uniqueMutations; // get the unique mutations for the individual
		uniqueNuc = unique[unique.mutationType == m1 | unique.mutationType == m2 | unique.mutationType == m3 | unique.mutationType == m4]; // subset to get only nuclear mutations
		uniqueMito = unique[unique.mutationType == m5 | unique.mutationType == m6 | unique.mutationType == m7 | unique.mutationType == m8]; // subset to get only mitochondrial mutations

We loop through each unique mitochondrial mutation in the individual. If epi has not been activated, we add to the global fitness the selection coefficient of the mutation squared while preserving its sign. If epi has been activated, then we gather all potentially epistatic nuclear mutations as determined by the epiSt argument. Mutations are epistatic if the position of the nuclear mutation % epiSt is equal to the position of the mitochondrial mutation % epiSt.

In [ ]:
// loop through each mitochondrial mutation to calculate the epistatic interactions with nuclear mutations
		for (m in uniqueMito) {
			if (!epi) {
				globalFitness = globalFitness + (m.selectionCoeff * abs(m.selectionCoeff));
			} else {
				total = 0;
				epistatic = uniqueNuc[uniqueNuc.position % epiSt == m.position % epiSt]; // get the nuclear mutations that are epistatic with the mitochondrial mutation (i.e., those that occur at the same position modulo 125)


We loop through the epistatic nuclear mutations to calculate the total effect on fitness. If the nuclear mutations are m1 or m3 (type A mutations), then the absoluate value of the selection coefficient is added to the total epistatic effect for this mitochondrial mutation site. If nuclear mutations are m2 or m4, the absolute value of the selection coefficient is subtracted from the total.

In [ ]:
// loop through the epistatic nuclear mutations to calculate the total effect on fitness
				for (n in epistatic) {
					if (n.mutationType == m1 | n.mutationType == m3) {
						total = total + abs(n.selectionCoeff);
					}
					else {
						total = total - abs(n.selectionCoeff);
					}
				}

If there are no epistatic nuclear mutations, the fitness is calculated normally as the selection coefficient squared while keeping the same sign. If the mitochondrial mutation is m6 or m8 (type B), then the total epistatic effect times the absolute values of the mitochondrial mutation's selection coefficient is subtracted from the current global fitness. Otherwise, if the mitochondrial mutation is type A, then the total epistatic effect times the absolute value of the selection coefficient is added to the current global fitness.

In [ ]:
                if (length(epistatic) == 0) {
					globalFitness = globalFitness + (m.selectionCoeff * abs(m.selectionCoeff));
				}

				if (m.mutationType == m6 | m.mutationType == m8) {
					globalFitness = globalFitness - (total * abs(m.selectionCoeff));
				}

				else 
					{globalFitness = globalFitness + (total * abs(m.selectionCoeff));
				}
			}
		}

For each individual, we loop through each nuclear mutation and add their effects on fitness independent of any epistatic effects.

In [ ]:
// loop through the nuclear mutations to add their effects on fitness (after accounting for epistasis with mitochondrial mutations)
	
        for (mut in uniqueNuc) {
			globalFitness = globalFitness + (abs(mut.selectionCoeff) * mut.selectionCoeff);
		}

For each individual, we scale their individual fitness according to a linear fitness to ensure that the majority of fitness values are between 0 and 2. We then add the individual fitness to an array of all individuals' fitness. FitnessScaling is applied to each individual.

In [ ]:
// calculate the final fitness for the individual, ensuring it is not negative
		Fitness = max(0.0, 1.0 + globalFitness * 3.0); // scale fitness to ensure it is positive and has a reasonable range
		popFitt = c(popFitt, Fitness);
	}

	inds.fitnessScaling = popFitt;
}

We set each mutationEffect to return 1.0 to deactivate SLiM's default mutation effect function and the previous calculations instead. This allows us to correctly account for epistasis.

In [ ]:
// Deactivate SLiM's default mutation effect function and use a custom one that allows for epistasis

mutationEffect(m1) {
	return 1.0;
}

mutationEffect(m2) {
	return 1.0;
}

mutationEffect(m3) {
	return 1.0;
}

mutationEffect(m4) {
	return 1.0;
}

mutationEffect(m5) {
	return 1.0;
}

mutationEffect(m6) {
	return 1.0;
}

mutationEffect(m7) {
	return 1.0;
}

mutationEffect(m8) {
	return 1.0;
}


After 4,000 generations, we end the simulation.The rest of this code creates files to store data based, named based on whether the sex, ben, and epi variables have been activated or not.

In [ ]:
// End the simulation after 4000 generations
4000 late() {
	if (sex) {
		S = "/sex_";
	} else {
		S = "/asex_";
	}
	
	if (ben) {
		B = "ben_";
	} else {
		B = "noben_";
	}
	
	if (epi) {
		Ep = "epi";
	} else {
		Ep = "noepi";
	}
	
	finalPath = getwd() + S + B + Ep + ".csv";

	values = sim.getValue("fitness_over_time");
	line = paste(values, sep=",");
	writeFile(finalPath, line, append=T);

	sim.simulationFinished();
}